# modular-diffusion: the layout writer by model size and by worked examples

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/modular-diffusion/blob/v0.13.0/colab/layout_sizes.ipynb)

The parts of a modular picture model are served by declared addresses: which things a picture holds, how many, which color on which, and where each goes on a 3 x 3 grid. A first measurement (`colab/layout_write.ipynb`) found that a 2B language model writes the addresses of one-statement prompts nearly right and those of composite prompts (two to five things with their counts, colors and places) right less than half the time. This notebook asks larger models of the same family, and worked examples of the composite kind (`geolip.modular_diffusion.layout`, `experiments.layout_write`); nothing is trained.

THE WRITERS: Qwen3.5-2B, Qwen3.5-4B and Qwen3.5-9B at pinned commits, and the LoRA fine-tune of each (`AbstractPhil/qwen3.5-2b-think-symbolic-image`, `AbstractPhil/qwen3.5-4b-think-symbolic-image`, `AbstractPhil/qwen3.5-9b-think-symbolic-image`) at the tag `lucia-v1.2`, whose commit is checked, merged into the weights. THE FORM: one line a thing `name | color | column | row`. THE WORKED EXAMPLES: six (one of each family), nine (the six followed by three of the composite kind), or none. THE PROMPTS: 600 prompts whose answers are known by construction (100 each of count, absent, colour, place, relation and combined, from a seeded generator) and the first captions of COCO val2017 rows 0 to 99. THE CELLS (16): at 2B six and nine examples, at 4B and 9B six, nine and none, each for the base model and its fine-tune; one user turn and no system turn; greedy decoding, thinking off, at most 512 new tokens, half precision. Every answer is read by the form's strict grammar and, beside it, by the table-tolerant read (a Markdown table's code fence, outer bars, header line and separator line taken off first). THE COMPARISONS (14, prompt by prompt): nine against six examples for each model, and the fine-tune against its base for each example set; the comparisons across sizes are made from the scored rows afterwards.

`SIZES` (at the top of the second and third cells) chooses the part a machine runs: `all`, `2b+4b` (fits a 16 GB card) or `9b` (needs about 20 GB or more); no job of one part needs a job of another, so two machines can each take one.

1. install at the release tag and print the environment;
2. the pins of the chosen sizes checked (each adapter's tag must resolve to its pinned commit) and the models fetched; the 100 real captions; the scorer's own controls on fake writers (its own layouts score 1, also with the nine examples; broken answers fail with their own kinds; the table-tolerant read changes nothing it should not); every chosen cell on 8 prompts a family and 8 captions with its seconds (for 9B the batch is the largest of 8, 4 and 2 that runs without running out of the card's memory); the projected minutes; the smoke shipped;
3. the chosen job list, one job at a time; each finished folder (a cell's raw answers, its scored rows, its summary and controls; a comparison's result) is shipped at once; a progress line a minute; the summary; the machine is given back only when every finished job and the report were shipped.

RESUME: run the cells again with the same `SIZES`. Finished jobs keep their markers under `OUT`. On a new machine, finished jobs are fetched back from the Hub with their files, so nothing runs twice.

In [ ]:
# C1 - install at the tag (extras on the name); torch is not a dependency, so the runtime's torch stays
%pip install -q "modular-diffusion[layout,hub,data] @ git+https://github.com/AbstractEyes/modular-diffusion@v0.13.0"
import importlib.metadata, subprocess, sys
try:                 # an old torchao beside PEFT stops PEFT from building a LoRA (PEFT wants 0.16 or newer, or none)
    _tao = importlib.metadata.version("torchao")
    if tuple(int(x) for x in _tao.split(".")[:2]) < (0, 16):
        subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])
        print("torchao", _tao, "removed: PEFT cannot build a LoRA beside it")
except (importlib.metadata.PackageNotFoundError, ValueError):
    pass
import os, sys, time, json, gc, hashlib, shutil, threading
import torch, geolip.modular_diffusion as MD
assert tuple(int(x) for x in MD.__version__.split(".")) >= (0, 13, 0), "RESTART REQUIRED: Runtime > Restart session, then run C1 again"
import transformers, huggingface_hub
try:
    import peft
    PEFT = peft.__version__
except ImportError:
    PEFT = "not installed (the plain merge is used)"
print("torch", torch.__version__, "| transformers", transformers.__version__, "| huggingface_hub", huggingface_hub.__version__,
      "| peft", PEFT, "| package", MD.__version__)
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("card:", props.name, f"| memory {props.total_memory / 2**30:.1f} GiB | cpus: {os.cpu_count()}")
else:
    print("card: NONE (choose a GPU runtime)")
OUT = "/content/layout_sizes"       # the output root; keep it the same to resume
os.makedirs(OUT, exist_ok=True)

Choose `SIZES` first: `all`, `2b+4b` or `9b`. Only the chosen sizes are fetched, smoked and run. Each base model comes from the Hub at its pinned commit; each adapter's tag must resolve to its pinned commit, else the cell stops. No cell has a system turn. The real prompts are the first captions of rows 0 to 99 of the first COCO val2017 parquet file. Then the scorer's own controls run on fake writers: the generator's layouts written in the line form must score 1 on every prompt with the six and with the nine examples, every deliberately broken answer must fail with its own kind, and the table-tolerant read must equal the strict read on those layouts and read them right when they are dressed as a Markdown table; a fake writer that answers every prompt with its true layout dressed as a table must read 0 by the strict grammar and 1 by the table-tolerant read on every family, and its prompt-by-prompt comparison with the plain true writer must differ by 1 under the strict read and by 0 under the tolerant one. Then every chosen cell runs on 8 prompts of each family and 8 real captions on this card, with its seconds and its repeat check; at 9B the batch starts at 8 and is halved (to 4, then 2) when the card runs out of memory, and the batch it settles on is printed and stored with the instruments; the chosen job list is projected from those seconds; the smoke folder is shipped.

In [ ]:
# C2 - the pins, the models, the captions, the controls, the smoke, the projection
SIZES = "all"  #@param ["all", "2b+4b", "9b"]
import re
from huggingface_hub import snapshot_download, hf_hub_download, HfApi
import pyarrow.parquet as pq
from geolip.modular_diffusion import hub, run, layout
from geolip.modular_diffusion.experiments import layout_write
PARTS = {"all": ("2b", "4b", "9b"), "2b+4b": ("2b", "4b"), "9b": ("9b",)}
assert SIZES in PARTS, f"SIZES is one of {sorted(PARTS)}, not {SIZES!r}"
CHOSEN, PART, PART_SIZES = PARTS[SIZES], SIZES.replace("+", "_"), SIZES     # the third cell checks that it runs this part
RUNS, SMOKE, REAL = f"{OUT}/runs", f"{OUT}/smoke_{PART}", f"{OUT}/real_captions.json"
REPO = "AbstractPhil/geolip-multirotary-diffusion-collective"   # the results repository ("" = nothing shipped)
PREFIX = "modular_diffusion/conditioning/address_writer_sizes"
NOT_SHIPPED = ("SHIPPED", "FROM_HUB")
os.makedirs(RUNS, exist_ok=True)
# a size: (the base model, its pinned commit, the fine-tune's adapter, the commit its tag must resolve to)
MODELS = {"2b": ("Qwen/Qwen3.5-2B", "15852e8c16360a2fea060d615a32b45270f8a8fc", "AbstractPhil/qwen3.5-2b-think-symbolic-image", "9366fbbe7846a7452c9dfc59cecfe46eca519c5a"),
          "4b": ("Qwen/Qwen3.5-4B", "851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a", "AbstractPhil/qwen3.5-4b-think-symbolic-image", "4c315a727f69ef2c777d4a58053dfb024304b261"),
          "9b": ("Qwen/Qwen3.5-9B", "c202236235762e1c871ad0ccb60c8ee5ba337b9a", "AbstractPhil/qwen3.5-9b-think-symbolic-image", "ecf2b9a5c3f8eb752fdb5041568c23e0f861129e")}
ADAPTER_TAG = "lucia-v1.2"
EXAMPLE_SETS = {"2b": (2, 1), "4b": (2, 1, 0), "9b": (2, 1, 0)}      # 2 the nine examples, 1 the six, 0 none
BASE = {s: f"{OUT}/base{s}" for s in MODELS}
ADAPTER = {s: f"{OUT}/lucia{s}" for s in MODELS}

# the pins and the fetches (the chosen sizes only)
api = HfApi()
ADAPTER_SHA = {}
for s in CHOSEN:
    base, rev, adapter, commit = MODELS[s]
    assert api.model_info(base, revision=rev).sha == rev, f"the {s} base model's commit"
    ADAPTER_SHA[s] = api.model_info(adapter, revision=ADAPTER_TAG).sha
    print(f"{s}: {base} at {rev}; {adapter} tag {ADAPTER_TAG} -> {ADAPTER_SHA[s]}")
    assert ADAPTER_SHA[s] == commit, f"the {s} adapter's tag {ADAPTER_TAG} does not resolve to the pinned commit {commit}: STOP"
for s in CHOSEN:
    base, rev, adapter, commit = MODELS[s]
    snapshot_download(repo_id=base, revision=rev, local_dir=BASE[s])
    snapshot_download(repo_id=adapter, revision=commit, local_dir=ADAPTER[s],
                      allow_patterns=["adapter_config.json", "adapter_model.safetensors"])
    assert all(os.path.exists(f"{ADAPTER[s]}/{f}") for f in ("adapter_config.json", "adapter_model.safetensors")), \
        f"the {s} adapter's files are not at the root of its repository: look at its file list"
PARQUET = hf_hub_download(repo_id="lmms-lab-encoder/COCO-Caption2017", repo_type="dataset", revision="3bdd5827e243cc3084ac69a1111e69c3ab9193ff",
                          filename="data/val-00000-of-00002.parquet", local_dir=f"{OUT}/coco")
captions = []      # the first caption of rows 0 to 99, read without the pictures (the rule of the capture notebooks)
for b in pq.ParquetFile(PARQUET).iter_batches(batch_size=128, columns=["answer"]):
    for a in b.column(0).to_pylist():
        captions.append("" if a is None else str((a[0] if a else "") if isinstance(a, (list, tuple)) else a))
    if len(captions) >= 100:
        break
captions = captions[:100]
assert len(captions) == 100 and all(c.strip() for c in captions), "the real captions"
json.dump(captions, open(REAL, "w", encoding="utf-8"), indent=1)
print("real captions:", len(captions), "| the first:", captions[0])

# the cells: (tag, size, arguments); one user turn, no system turn, the line form only
DEVICE = "cuda"
DTYPE = "bf16" if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else "fp16"     # half precision
BATCH = {"2b": 8, "4b": 8, "9b": 8}      # 9b: the largest of 8, 4, 2 that runs the smoke (settled below)
WRITERS = {s: {"base": f"writer={BASE[s]}", "lucia": f"writer={BASE[s]} adapter={ADAPTER[s]}"} for s in MODELS}
COMMON = f"device={DEVICE} dtype={DTYPE} max_new_tokens=512 local_only=1 seed=0 threads=4"
CELLS = [(layout_write.cell_name(f"{w}{s}", "cell", ex), s, f"{WRITERS[s][w]} form=cell examples={ex}")
         for s in CHOSEN for w in ("base", "lucia") for ex in EXAMPLE_SETS[s]]
# the comparisons inside a size: nine against six examples for each model; the fine-tune against its base per example set
COMPARES = []
for s in CHOSEN:
    COMPARES += [(f"cmp_{w}{s}_cell_ex2_vs_ex1", s, f"{w}{s}_cell_ex2", f"{w}{s}_cell_ex1") for w in ("base", "lucia")]
    COMPARES += [(f"cmp_lucia{s}_vs_base{s}_cell_ex{ex}", s, f"lucia{s}_cell_ex{ex}", f"base{s}_cell_ex{ex}")
                 for ex in EXAMPLE_SETS[s]]
assert len({t for t, *_ in CELLS + COMPARES}) == len(CELLS) + len(COMPARES)
assert SIZES != "all" or (len(CELLS) == 16 and len(COMPARES) == 14)

def job_list(runs, extra=""):
    # a size's cells, then its comparisons (each waits for its two cells); the batch of a size as settled by the smoke
    jobs = []
    for s in CHOSEN:
        jobs += [(t, f"variant=write {a} batch={BATCH[s]} {COMMON} real={REAL} repeat_n=20{extra}", [], "cell")
                 for t, ss, a in CELLS if ss == s]
        jobs += [(t, f"variant=write_compare a={runs}/{x} b={runs}/{y}", [x, y], "compare")
                 for t, ss, x, y in COMPARES if ss == s]
    return jobs

# the smoke: the scorer's own controls on fake writers (all 600 prompts, the six and the nine examples)
shutil.rmtree(SMOKE, ignore_errors=True)
for ex in (1, 2):
    r = run.main("layout_write", f"variant=write writer=fake:truth form=cell examples={ex} repeat_n=20".split()
                 + [f"out={SMOKE}/fake_truth_cell_ex{ex}"])
    fam = r["summary"]["families"]
    assert all(fam[f]["scores"]["address_right"]["mean"] == 1.0 for f in layout.FAMILIES), f"control (a), examples {ex}"
    for k in ("a", "c", "e"):
        assert r["controls"][k]["holds"] is True, f"control ({k}), examples {ex}"
    print(f"controls (a), (c), (e), examples {ex}: the generator's layouts score 1 on {r['controls']['a']['prompts']} prompts; "
          "every broken answer fails with its kind; the table-tolerant read equals the strict read and reads the dressed layouts")
# control (e), its third part, on the whole path of a cell (rows, summary, comparison): every prompt answered with its
# true layout dressed as a Markdown table (a fence, a header, a separator, a bar at each end of every row)
TAB = run.main("layout_write", "variant=write writer=fake:table form=cell examples=1 repeat_n=20".split()
               + [f"out={SMOKE}/fake_table_cell_ex1"])
for f in layout.FAMILIES:     # the strict read refuses every dressed answer, absent's '| none |' too (parsed 0: address right 0)
    assert TAB["summary"]["families"][f]["scores"]["address_right"]["mean"] == 0.0, f"control (e), the dressed writer, {f}"
    assert TAB["summary"]["families"][f]["scores"]["parsed"]["mean"] == 0.0, f"control (e), the dressed writer, {f}"
    assert TAB["summary"]["table"]["families"][f]["scores"]["address_right"]["mean"] == 1.0, f"control (e), the dressed writer, {f}"
assert TAB["controls"]["e"]["holds"] is True, "control (e) on the dressed writer"
CMP = run.main("layout_write", f"variant=write_compare a={SMOKE}/fake_table_cell_ex1 b={SMOKE}/fake_truth_cell_ex1".split()
               + [f"out={SMOKE}/cmp_fake_table_vs_truth_cell_ex1"])
for f in list(layout.FAMILIES) + ["all"]:     # 1 apart on every prompt by the strict read, 0 by the tolerant one
    x, y = CMP["scores"]["address_right"][f], CMP["table"]["address_right"][f]
    assert x["mean"] == -1.0 and x["tie"] is False and y["mean"] == 0.0 and y["tie"] is True, f"control (e), the comparison, {f}"
print("control (e), the dressed writer: strict address right 0 and tolerant 1 on all six families (600 prompts); against "
      "the plain true writer -1 by the strict read and 0 by the tolerant one, prompt by prompt")
# then every chosen cell on 8 prompts a family; at 9B the batch is halved while the card runs out of memory
SMOKE_N = 8
SEC, BATCH_TRIALS = {}, []
for t, s, a in sorted(CELLS, key=lambda c: (CHOSEN.index(c[1]), c[0].startswith("lucia"), -int(c[0][-1]))):
    while True:
        t0 = time.time()
        try:
            r = run.main("layout_write", f"variant=write {a} batch={BATCH[s]} {COMMON} real={REAL} first_n={SMOKE_N} "
                                         f"repeat_n={BATCH[s]}".split() + [f"out={SMOKE}/{t}"])
        except torch.cuda.OutOfMemoryError:
            r = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        BATCH_TRIALS.append({"cell": t, "batch": BATCH[s], "ran": r is not None, "seconds": time.time() - t0})
        if r is not None:
            break
        print(f"{t}: out of the card's memory at batch {BATCH[s]}", flush=True)
        assert s == "9b" and BATCH[s] > 2, f"{t}: out of the card's memory at batch {BATCH[s]}: this card cannot run it"
        BATCH[s] //= 2
        shutil.rmtree(f"{SMOKE}/{t}", ignore_errors=True)
    SEC[t] = {"seconds": time.time() - t0, "loading": r["seconds_loading"], "writing": r["seconds_writing"],
              "answers": r["prompts"] + (r["controls"]["d"].get("asked") or 0), "batch": BATCH[s]}
    tab = r["summary"]["table"]["all"]["scores"]["address_right"]["mean"]
    print(f"{t}: {r['prompts']} prompts in {SEC[t]['seconds']:.0f} s (loading {r['seconds_loading']:.0f} s, batch {BATCH[s]}); "
          f"parsed {r['summary']['all']['scores']['parsed']['mean']:.2f}; address right {r['summary']['all']['scores']['address_right']['mean']:.2f} "
          f"(table read {tab:.2f}); repeat {r['controls']['d'].get('same')}/{r['controls']['d'].get('compared')}", flush=True)
print("the batch a size:", {s: BATCH[s] for s in CHOSEN})

# the projection: each cell's seconds an answer (its writing plus the repeat, over the answers it gave) times the whole
# cell's answers (700 prompts and the first 20 asked again in whole batches), plus its loading
COST = {}
for t, s, _ in CELLS:
    per = (SEC[t]["seconds"] - SEC[t]["loading"]) / max(1, SEC[t]["answers"])
    COST[t] = SEC[t]["loading"] + 30 + per * (600 + 100 + -(-20 // BATCH[s]) * BATCH[s])
for t, *_ in COMPARES:
    COST[t] = 15.0
JOBS = job_list(RUNS)
INSTRUMENTS = {"sizes": SIZES, "chosen": list(CHOSEN), "smoke_seconds": SEC, "projected_seconds": COST,
               "card": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu", "dtype": DTYPE,
               "batch": {s: BATCH[s] for s in CHOSEN}, "batch_trials": BATCH_TRIALS, "adapter_tag": ADAPTER_TAG,
               "adapter_commits": ADAPTER_SHA, "models": {s: MODELS[s] for s in CHOSEN}, "real_captions": len(captions),
               "dressed_writer": {"strict": {f: TAB["summary"]["families"][f]["scores"]["address_right"]["mean"] for f in layout.FAMILIES},
                                  "tolerant": {f: TAB["summary"]["table"]["families"][f]["scores"]["address_right"]["mean"]
                                               for f in layout.FAMILIES},
                                  "against_truth": {"strict": CMP["scores"]["address_right"]["all"], "tolerant": CMP["table"]["address_right"]["all"]}},
               "versions": {"torch": torch.__version__, "transformers": transformers.__version__, "peft": PEFT,
                            "package": MD.__version__}}
json.dump(INSTRUMENTS, open(f"{SMOKE}/instruments.json", "w", encoding="utf-8"), indent=1)
print(f"the job list for SIZES {SIZES}: {len(JOBS)} jobs ({len(CELLS)} cells, {len(COMPARES)} comparisons), about "
      f"{sum(COST.values()) / 60:.0f} projected minutes on this card with one runner")
if REPO:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")          # placed in this process's environment only
    assert hub.ship(SMOKE, REPO, f"{PREFIX}/_smoke_{PART}", f"address writer sizes: smoke run ({SIZES})", ignore=NOT_SHIPPED), (
        "the smoke folder was not shipped: check the token and the repository before the long run")

Keep `SIZES` as in the second cell. One runner works the chosen job list in order: a size's cells, then its prompt-by-prompt comparisons (each waits for its two cells). Each finished job folder is shipped at once to `REPO` (a cell with its raw answers `outputs.jsonl`, its scored rows `rows.jsonl`, its `result.json` and `result.md`), with the token from the notebook's secret `HF_TOKEN` (placed in this process's environment only; the jobs themselves run without it). At the end the summary, the roster, its log and the instruments are shipped to `_report_<part>`. The machine is given back (`RELEASE`) only when the report and every finished job were shipped; if an upload failed, the machine is kept and the cell names the jobs.

In [ ]:
# C3 - the chosen job list, each finished job shipped, a progress line a minute, the summary, the machine
SIZES = "all"  #@param ["all", "2b+4b", "9b"]
assert SIZES == PART_SIZES, f"the second cell ran SIZES = {PART_SIZES!r}: choose the same here, or run the second cell again"
from geolip.modular_diffusion import jobs
RELEASE = True                     # give the machine back at the end (only when every finished job and the report were shipped)
EXTRA = ""                         # more arguments for every cell (empty in a real run)
PROGRESS_S = 60
JOBS = job_list(RUNS, EXTRA)
kind = {t: k for t, _, _, k in JOBS}
needs = {t: n for t, _, n, _ in JOBS}
tags = [t for t, *_ in JOBS]
roster = f"{OUT}/roster.txt"
cap = lambda t: max(30, int(3 * COST[t] / 60))
with open(roster, "w", encoding="utf-8", newline="\n") as fh:
    fh.write("\n".join(f"{t} | {cap(t)} | geolip.modular_diffusion.run | layout_write {a}" + (f" | needs={','.join(n)}" if n else "")
                       for t, a, n, _ in JOBS) + "\nEND\n")
mark = lambda t, m: os.path.exists(f"{RUNS}/{t}/{m}")
if REPO:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
CELL_FILES = ("outputs.jsonl", "rows.jsonl", "result.json")

def fetch_back():
    # on a new machine: finished jobs come back with their files, so nothing runs twice; a job comes back only when what
    # it reads came back too
    try:
        got = snapshot_download(repo_id=REPO, allow_patterns=[f"{PREFIX}/{t}/*" for t in tags], local_dir=f"{OUT}/hub_back")
    except Exception as e:
        print("nothing fetched back from the Hub:", repr(e)[:200])
        return
    for t in tags:
        src, dst = f"{got}/{PREFIX}/{t}", f"{RUNS}/{t}"
        if mark(t, "DONE") or not os.path.exists(f"{src}/result.json"):
            continue
        if kind[t] == "cell" and not all(os.path.exists(f"{src}/{f}") for f in CELL_FILES):
            continue                                             # its answers are not on the Hub: it is run again
        if not all(mark(n, "DONE") for n in needs[t]):
            continue
        shutil.copytree(src, dst, dirs_exist_ok=True)
        open(f"{dst}/DONE", "w").write("fetched back from the Hub\n")
        open(f"{dst}/FROM_HUB", "w").write("fetched back from the Hub\n")

if REPO:
    fetch_back()
for t in tags:      # a claim left by an interrupted session would hold its job forever
    if mark(t, "CLAIM") and not (mark(t, "DONE") or mark(t, "FAILED")):
        os.remove(f"{RUNS}/{t}/CLAIM")
def after_job(tag, folder, ok):
    if ok and REPO:
        if hub.ship(folder, REPO, f"{PREFIX}/{tag}", f"address writer sizes: {tag}", ignore=NOT_SHIPPED):
            open(f"{folder}/SHIPPED", "w").write("shipped to the Hub\n")      # kept across sessions on this machine
        else:
            print(f"{tag}: NOT SHIPPED (its folder stays on this machine: {folder})", flush=True)

shipped_or_fetched = lambda t: mark(t, "SHIPPED") or mark(t, "FROM_HUB")
for t in tags:      # a job that finished in an earlier session but whose upload failed: shipped again now
    if REPO and mark(t, "DONE") and not shipped_or_fetched(t):
        after_job(t, f"{RUNS}/{t}", True)

t0 = time.time()
left0 = sum(COST[t] for t in tags if not (mark(t, "DONE") or mark(t, "FAILED")))
print(f"start (SIZES {SIZES}): {sum(mark(t, 'DONE') for t in tags)}/{len(tags)} jobs already done; about {left0 / 60:.0f} min to run; one runner")
runner = threading.Thread(target=jobs.run_roster, daemon=True,
                          kwargs=dict(roster=roster, out=RUNS, stop=f"{OUT}/STOP", log=f"{OUT}/roster.log", card_room_mib=None,
                                      ram_free_gb=4.0, max_hours=30.0, claim=True, after_job=after_job, wait_s=5.0,
                                      env={"OMP_NUM_THREADS": "4", "MKL_NUM_THREADS": "4", "HF_HUB_OFFLINE": "1"}))
runner.start()
while runner.is_alive():
    time.sleep(PROGRESS_S)
    done, failed = sum(mark(t, "DONE") for t in tags), sum(mark(t, "FAILED") for t in tags)
    left = sum(COST[t] for t in tags if not (mark(t, "DONE") or mark(t, "FAILED")))
    spent = (time.time() - t0) / 60
    rate = (left0 - left) / 60 / spent if left0 > left and spent > 0 else 1.0     # projected minutes done a minute
    print(f"jobs done {done}/{len(tags)} (failed {failed}); {spent:.0f} min spent; about {left / 60 / rate:.0f} min left", flush=True)

summary = layout_write.summarize(RUNS)
print(summary)
REPORT = f"{OUT}/report_{PART}"
os.makedirs(REPORT, exist_ok=True)
open(f"{REPORT}/summary.md", "w", encoding="utf-8").write(summary)
json.dump(INSTRUMENTS, open(f"{REPORT}/instruments.json", "w", encoding="utf-8"), indent=1)
for f in ("roster.txt", "roster.log"):
    if os.path.exists(f"{OUT}/{f}"):
        shutil.copy(f"{OUT}/{f}", f"{REPORT}/{os.path.basename(f).replace('.log', '_log.txt')}")
for t in tags:      # the logs of the jobs that failed
    if mark(t, "FAILED"):
        for f in ("run.log", "log.txt"):
            if os.path.exists(f"{RUNS}/{t}/{f}"):
                shutil.copy(f"{RUNS}/{t}/{f}", f"{REPORT}/failed_{t}_{f.replace('.', '_')}.txt")
shipped = hub.ship(REPORT, REPO, f"{PREFIX}/_report_{PART}", f"address writer sizes: summary, roster and instruments ({SIZES})") if REPO else None
unshipped = [t for t in tags if mark(t, "DONE") and not shipped_or_fetched(t)]
print("report shipped:", shipped, "| finished jobs not on the Hub:", ", ".join(unshipped) or "none")
if RELEASE and shipped and not unshipped:
    from google.colab import runtime
    runtime.unassign()
else:
    print("THE MACHINE IS KEPT:", "the report was not shipped" if not shipped else
          f"these finished jobs were not shipped: {', '.join(unshipped)} (run C3 again to retry)" if unshipped else "RELEASE is False")